# Práctica con datos de estilo real: sobreajuste frente a subajuste

**Validación y ajuste de modelos, Tema 1**

En el cuaderno de tareas los datos se generaban dentro del propio cuaderno y la relación verdadera era conocida. En la práctica, el analista recibe un **archivo** y no conoce ni la relación verdadera ni el nivel de ruido. Este cuaderno reproduce esa situación con tres archivos CSV, cada uno construido en torno a una historia de negocio distinta y a una causa distinta de sobreajuste.

| Caso | Archivo | Tarea | Perilla de complejidad | Puntos |
|---|---|---|---|---|
| 1 | `data/01_ice_cream_sales.csv` | Regresión, 1 variable | grado del polinomio | 30 |
| 2 | `data/01_exam_scores.csv` | Regresión, 20 variables, 50 filas | número de variables | 30 |
| 3 | `data/01_customer_churn.csv` | Clasificación, variables de distinto tipo | profundidad del árbol de decisión | 40 |

> **Nota sobre los nombres.** Los archivos CSV conservan los nombres de columna originales en inglés (así aparecerían en un conjunto de datos real). Al comienzo de cada caso se incluye un glosario con su significado en español.

### Instrucciones

1. Ejecute primero la celda de **Configuración**. Localiza la carpeta `data/` y carga las comprobaciones automáticas.
2. Dentro de cada caso, resuelva los apartados **en orden**; los tres casos son independientes entre sí.
3. Sustituya cada `None` marcado con `# TODO` por el código adecuado y ejecute la celda `check_...()`: ✅ correcto, ❌ incorrecto (con una pista), ⏳ todavía sin hacer.
4. Las celdas marcadas como **Respuesta** requieren una breve explicación escrita y las califica el instructor.
5. El conjunto de prueba se usa **una sola vez por caso**, para la evaluación final. Todas las decisiones sobre el modelo se toman con los datos de entrenamiento y la validación cruzada.

## Configuración *(ejecute esta celda; no la modifique)*

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold, cross_val_score, train_test_split, validation_curve
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.tree import DecisionTreeClassifier

SEED = 0
CV = KFold(5, shuffle=True, random_state=SEED)
BLUE, GREEN, ORANGE, RED = "#0066CC", "#00994C", "#FF6600", "#CC0033"
plt.rcParams.update({"figure.figsize": (8, 4.5), "axes.grid": True, "grid.alpha": 0.3})

# El cuaderno funciona desde la carpeta del tema y desde la subcarpeta solutions/
DATA_DIR = next(p for p in [Path("data"), Path("../data")] if p.exists())
print("Carpeta de datos:", DATA_DIR.resolve())
print("Archivos:", sorted(f.name for f in DATA_DIR.glob("*.csv")))


def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def poly_model(degree):
    return make_pipeline(StandardScaler(), PolynomialFeatures(degree),
                         StandardScaler(), LinearRegression())


def _ok(m):   print("✅", m)
def _bad(m):  print("❌", m)
def _todo(m): print("⏳", m)

---
# Caso 1. Ventas de helados y temperatura (30 puntos)

Un quiosco de playa registró sus ventas diarias de helados durante 36 días, junto con la temperatura máxima de cada día. El dueño quiere un modelo que prediga las ventas a partir del pronóstico del tiempo para planificar el inventario.

**Glosario de columnas**

| Columna | Significado |
|---|---|
| `date` | Fecha de la observación |
| `max_temperature_c` | Temperatura máxima del día (°C) |
| `sales_units` | Número de helados vendidos (variable objetivo) |

### 1.1 Cargar y explorar (5 puntos)

Cargue el archivo en `ice`, muestre sus primeras filas y sus estadísticas descriptivas, y dibuje un diagrama de dispersión de `sales_units` frente a `max_temperature_c`.

In [ ]:
ice = None   # TODO: leer DATA_DIR / "01_ice_cream_sales.csv"

# TODO: mostrar ice.head() e ice.describe()

# TODO: diagrama de dispersión de las ventas frente a la temperatura

**Pregunta escrita.** Describa la forma de la relación en el diagrama de dispersión. ¿Sería una línea recta un modelo adecuado? ¿Qué cabría esperar que ocurriera con las ventas en días extremadamente calurosos?

**Respuesta:** *(escriba aquí la respuesta)*

### 1.2 Tres modelos candidatos (10 puntos)

1. Separe los datos en `X_ice` (un DataFrame con la columna `max_temperature_c`) e `y_ice` (`sales_units`), y después en 70 % de entrenamiento y 30 % de prueba con `random_state=SEED`.
2. Ajuste `poly_model(d)` para `d` igual a **1, 5 y 15** con los datos de entrenamiento.
3. Guarde el RMSE de entrenamiento y de prueba en el DataFrame `ice_results` (índice: grado; columnas: `train_rmse`, `test_rmse`).

In [ ]:
X_ice, y_ice = None, None                                # TODO
Xi_train, Xi_test, yi_train, yi_test = None, None, None, None   # TODO

ice_models = {}
rows = []
for d in [1, 5, 15]:
    # TODO: ajustar poly_model(d), guardarlo en ice_models[d] y añadir un dict a rows
    pass

ice_results = None   # TODO: DataFrame indexado por el grado
ice_results

In [ ]:
# Gráfico (ya completo)
if ice_results is not None:
    grid = pd.DataFrame({"max_temperature_c": np.linspace(ice["max_temperature_c"].min(),
                                                          ice["max_temperature_c"].max(), 300)})
    fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
    for ax, (d, c) in zip(axes, [(1, ORANGE), (5, GREEN), (15, RED)]):
        ax.scatter(Xi_train, yi_train, color=BLUE, s=25, label="entrenamiento")
        ax.scatter(Xi_test, yi_test, color=RED, marker="x", s=30, label="prueba")
        ax.plot(grid, ice_models[d].predict(grid), color=c, lw=2.5)
        ax.set_ylim(-50, 500)
        ax.set_title(f"grado {d}: entren. {ice_results.loc[d, 'train_rmse']:.0f} / "
                     f"prueba {ice_results.loc[d, 'test_rmse']:.0f}", color=c, fontweight="bold")
        ax.set_xlabel("temperatura (°C)")
    axes[0].set_ylabel("ventas"); axes[0].legend()
    plt.show()


def check_1_2():
    if ice_results is None:
        return _todo("1.2 todavía sin completar")
    a, b, c, d_ = train_test_split(ice[["max_temperature_c"]], ice["sales_units"], test_size=0.3, random_state=SEED)
    if len(Xi_train) != len(a) or not Xi_train.index.equals(a.index):
        return _bad("la división no es correcta (test_size=0.3, random_state=SEED)")
    for d in [1, 5, 15]:
        m = poly_model(d).fit(a, c)
        if not np.isclose(ice_results.loc[d, "test_rmse"], rmse(d_, m.predict(b)), rtol=1e-3):
            return _bad(f"grado {d}: el RMSE de prueba no coincide. ¿Se ajustó el modelo con los datos de entrenamiento?")
        if not np.isclose(ice_results.loc[d, "train_rmse"], rmse(c, m.predict(a)), rtol=1e-3):
            return _bad(f"grado {d}: el RMSE de entrenamiento no coincide")
    _ok("1.2 correcto")

check_1_2()

### 1.3 Elegir el grado sin usar el conjunto de prueba (10 puntos)

1. Calcule una curva de validación con los datos de **entrenamiento** para los grados 1 a 15, con `cv=CV` y `scoring="neg_root_mean_squared_error"`. Guarde en `ice_val_rmse` la **mediana** del RMSE de validación (en positivo) a través de los pliegues.
2. Guarde en `ice_best_degree` el grado con el menor RMSE de validación.
3. Ajuste el modelo elegido con todo el conjunto de entrenamiento y guarde su RMSE de prueba en `ice_final_test_rmse`.

In [ ]:
ice_degrees = np.arange(1, 16)

ice_val_rmse = None          # TODO
ice_best_degree = None       # TODO
ice_final_test_rmse = None   # TODO
print("mejor grado:", ice_best_degree, "| RMSE final de prueba:", ice_final_test_rmse)

In [ ]:
if ice_val_rmse is not None:
    fig, ax = plt.subplots()
    ax.plot(ice_degrees, ice_val_rmse, "s-", color=RED, label="RMSE de validación (mediana de 5 pliegues)")
    ax.axvline(ice_best_degree, color=GREEN, ls="--", label=f"mejor grado = {ice_best_degree}")
    ax.set_yscale("log")
    ax.set(xlabel="grado", ylabel="RMSE (escala logarítmica)", title="Caso 1: curva de validación")
    ax.legend(); plt.show()


def check_1_3():
    if ice_final_test_rmse is None:
        return _todo("1.3 todavía sin completar")
    _, va = validation_curve(poly_model(1), Xi_train, yi_train, param_name="polynomialfeatures__degree",
                             param_range=np.arange(1, 16), cv=CV, scoring="neg_root_mean_squared_error")
    exp = -np.median(va, axis=1)
    if np.any(np.asarray(ice_val_rmse) < 0):
        return _bad("ice_val_rmse es negativo: cambie el signo de las puntuaciones")
    if not np.allclose(ice_val_rmse, exp, rtol=1e-3):
        return _bad("ice_val_rmse no coincide: ¿datos de entrenamiento, cv=CV y mediana sobre axis=1?")
    best = int(np.arange(1, 16)[np.argmin(exp)])
    if ice_best_degree != best:
        return _bad(f"ice_best_degree debería ser {best}")
    exp_te = rmse(yi_test, poly_model(best).fit(Xi_train, yi_train).predict(Xi_test))
    if not np.isclose(ice_final_test_rmse, exp_te, rtol=1e-3):
        return _bad(f"el RMSE final de prueba debería ser {exp_te:.2f}")
    _ok(f"1.3 correcto: grado {best}, RMSE final de prueba {exp_te:.1f}")

check_1_3()

**Pregunta escrita (5 puntos).** Diagnostique los modelos de grado 1 y de grado 15 a partir de `ice_results`. En este problema de estilo real el nivel de ruido es desconocido: ¿qué evidencia, distinta de un nivel de ruido conocido, respalda cada diagnóstico?

**Respuesta:** *(escriba aquí la respuesta)*

---
# Caso 2. Notas de examen y demasiadas variables (30 puntos)

Un profesor recogió datos de 50 estudiantes: cuatro variables relacionadas con el estudio y dieciséis respuestas a una encuesta general (talla de zapato, número favorito, número de mascotas, etc.). El profesor quiere saber qué variables predicen la nota del examen final.

Con 20 variables y solo 35 filas de entrenamiento, una regresión lineal tiene muchísima libertad y puede usar columnas irrelevantes para «explicar» variaciones casuales de la muestra de entrenamiento. Se trata de sobreajuste causado por **demasiadas variables en relación con el número de observaciones**.

**Glosario de columnas**

| Columna | Significado |
|---|---|
| `student_id` | Identificador (no es una variable predictora) |
| `study_hours_week` | Horas de estudio por semana |
| `sleep_hours_night` | Horas de sueño por noche (promedio) |
| `attendance_pct` | Asistencia a clase (%) |
| `previous_exam_score` | Nota del examen anterior (0–100) |
| `shoe_size_eu` | Talla de zapato (europea) |
| `birth_month` | Mes de nacimiento |
| `favorite_number` | Número favorito |
| `height_cm` | Estatura (cm) |
| `pets_owned` | Número de mascotas |
| `siblings` | Número de hermanos |
| `distance_home_km` | Distancia de casa a la escuela (km) |
| `locker_number` | Número de taquilla |
| `bus_route` | Ruta de autobús |
| `seat_row` | Fila del asiento en clase |
| `hair_length_cm` | Largo del cabello (cm) |
| `steps_yesterday` | Pasos caminados ayer |
| `music_hours_week` | Horas de música por semana |
| `social_media_min_day` | Minutos diarios en redes sociales |
| `breakfast_days_week` | Días por semana que desayuna |
| `phone_battery_pct` | Batería del teléfono al responder (%) |
| `final_exam_score` | Nota del examen final (variable objetivo) |

Las columnas desde `shoe_size_eu` hasta `phone_battery_pct` son las dieciséis respuestas de la encuesta general.

### 2.1 Cargar e inspeccionar las correlaciones (8 puntos)

1. Cargue el archivo en `exam`.
2. Construya `X_exam` con **todas las columnas excepto** `student_id` y `final_exam_score`, e `y_exam` con `final_exam_score`.
3. Calcule la correlación absoluta de cada variable con el objetivo, ordenada de mayor a menor, y guárdela en `exam_corr` (una Series).

In [ ]:
exam = None             # TODO
X_exam, y_exam = None, None   # TODO
exam_corr = None        # TODO: X_exam.corrwith(y_exam).abs().sort_values(ascending=False)
exam_corr

In [ ]:
if exam_corr is not None:
    fig, ax = plt.subplots(figsize=(8, 6))
    colours = [GREEN if c in ["study_hours_week", "sleep_hours_night", "attendance_pct",
                              "previous_exam_score"] else "gray" for c in exam_corr.index[::-1]]
    ax.barh(exam_corr.index[::-1], exam_corr.values[::-1], color=colours)
    ax.set(xlabel="|correlación con final_exam_score|", title="Caso 2: correlación de cada variable con la nota")
    plt.show()


def check_2_1():
    if exam_corr is None:
        return _todo("2.1 todavía sin completar")
    if X_exam.shape != (50, 20):
        return _bad(f"X_exam debería tener 50 filas y 20 columnas; tiene {X_exam.shape}")
    if "student_id" in X_exam or "final_exam_score" in X_exam:
        return _bad("X_exam no debe contener student_id ni final_exam_score")
    if not exam_corr.is_monotonic_decreasing or (exam_corr < 0).any():
        return _bad("exam_corr debe contener correlaciones ABSOLUTAS ordenadas de mayor a menor")
    _ok(f"2.1 correcto: la variable más fuerte es {exam_corr.index[0]}")

check_2_1()

### 2.2 Todas las variables frente a las cuatro de estudio (12 puntos)

1. Divida `X_exam`, `y_exam` en 70 % de entrenamiento y 30 % de prueba con `random_state=SEED`.
2. Ajuste dos pipelines `make_pipeline(StandardScaler(), LinearRegression())`:
   - el **modelo A** usa las 20 variables;
   - el **modelo B** usa solo `STUDY_FEATURES`.
3. Guarde el $R^2$ de entrenamiento, el $R^2$ de prueba y el $R^2$ de validación cruzada con 5 pliegues sobre el conjunto de entrenamiento (`cross_val_score(..., cv=CV, scoring="r2").mean()`) en el DataFrame `exam_results`, indexado por `"A: las 20"` y `"B: 4 de estudio"`, con las columnas `train_r2`, `cv_r2`, `test_r2`.

In [ ]:
STUDY_FEATURES = ["study_hours_week", "sleep_hours_night", "attendance_pct", "previous_exam_score"]

Xe_train, Xe_test, ye_train, ye_test = None, None, None, None   # TODO

rows = []
for name, cols in [("A: las 20", list(X_exam.columns) if X_exam is not None else []),
                   ("B: 4 de estudio", STUDY_FEATURES)]:
    # TODO: construir el pipeline, calcular train_r2, cv_r2 y test_r2, y añadir un dict a rows
    pass

exam_results = None   # TODO
exam_results

In [ ]:
def check_2_2():
    if exam_results is None:
        return _todo("2.2 todavía sin completar")
    a, b, c, d = train_test_split(X_exam, y_exam, test_size=0.3, random_state=SEED)
    exp = {}
    for name, cols in [("A: las 20", list(X_exam.columns)), ("B: 4 de estudio", STUDY_FEATURES)]:
        p = make_pipeline(StandardScaler(), LinearRegression())
        cv = cross_val_score(p, a[cols], c, cv=CV, scoring="r2").mean()
        p.fit(a[cols], c)
        exp[name] = (p.score(a[cols], c), cv, p.score(b[cols], d))
    for name, (tr, cv, te) in exp.items():
        if name not in exam_results.index:
            return _bad(f"falta la fila '{name}' en exam_results (use exactamente esos nombres de índice)")
        got = exam_results.loc[name, ["train_r2", "cv_r2", "test_r2"]].to_numpy(dtype=float)
        if not np.allclose(got, [tr, cv, te], rtol=1e-3):
            return _bad(f"{name}: se esperaba train={tr:.3f}, cv={cv:.3f}, test={te:.3f}")
    _ok("2.2 correcto")

check_2_2()

**Pregunta escrita (10 puntos).**
(a) ¿Qué modelo tiene el $R^2$ de *entrenamiento* más alto y cuál el $R^2$ de *prueba* más alto? Explique la contradicción.
(b) ¿Por qué añadir una columna inútil como `favorite_number` nunca puede bajar el $R^2$ de entrenamiento de una regresión lineal?
(c) ¿Habría llevado a la elección correcta el $R^2$ de validación cruzada por sí solo, sin mirar el conjunto de prueba?

**Respuesta:** *(escriba aquí la respuesta)*

---
# Caso 3. Abandono de clientes (40 puntos)

Una empresa de suscripciones quiere predecir qué clientes **cancelarán (abandono, *churn*)** en los próximos tres meses, para ofrecerles un descuento de retención. El archivo contiene 900 clientes.

**Glosario de columnas**

| Columna | Significado |
|---|---|
| `customer_id` | Identificador (no es una variable predictora) |
| `age` | Edad del cliente |
| `contract_type` | Tipo de contrato: `month-to-month` (mes a mes), `one-year` (un año) o `two-year` (dos años) (categórica) |
| `payment_method` | Forma de pago: `credit card` (tarjeta de crédito), `bank transfer` (transferencia bancaria) o `electronic check` (cheque electrónico) (categórica) |
| `tenure_months` | Antigüedad como cliente (meses) |
| `monthly_charges` | Factura mensual |
| `support_calls_90d` | Llamadas a atención al cliente en los últimos 90 días |
| `churned` | 1 = canceló, 0 = se quedó (variable objetivo) |

### 3.1 Carga, línea base y codificación (10 puntos)

1. Cargue el archivo en `churn`.
2. Calcule la **exactitud de la línea base** `churn_baseline`: la exactitud de predecir siempre la clase mayoritaria.
3. Construya `X_churn` eliminando `customer_id` y `churned`, y aplique **codificación one-hot** a las dos columnas categóricas con `pd.get_dummies(..., drop_first=True, dtype=int)`. Construya `y_churn` a partir de `churned`.

In [ ]:
churn = None             # TODO
churn_baseline = None    # TODO: proporción de la clase mayoritaria
X_churn, y_churn = None, None   # TODO
print("exactitud de la línea base:", churn_baseline)
X_churn.head() if X_churn is not None else None

In [ ]:
def check_3_1():
    if X_churn is None or churn_baseline is None:
        return _todo("3.1 todavía sin completar")
    if not np.isclose(churn_baseline, 1 - churn["churned"].mean()):
        return _bad("churn_baseline debe ser la proporción de la clase MAYORITARIA (clientes que se quedaron)")
    if X_churn.shape != (900, 8):
        return _bad(f"X_churn debería ser 900 x 8 tras la codificación one-hot con drop_first=True; es {X_churn.shape}")
    if X_churn.select_dtypes("object").shape[1] > 0:
        return _bad("X_churn todavía contiene columnas de texto: codifíquelas con pd.get_dummies")
    _ok(f"3.1 correcto: exactitud de la línea base = {churn_baseline:.3f}, {X_churn.shape[1]} variables")

check_3_1()

### 3.2 Un árbol sin restricciones (8 puntos)

1. Divida en 75 % de entrenamiento y 25 % de prueba con `stratify=y_churn` y `random_state=SEED`.
2. Ajuste `DecisionTreeClassifier(random_state=SEED)` **sin** ningún límite de profundidad.
3. Guarde su profundidad en `full_tree_depth`, su exactitud de entrenamiento en `full_tree_train_acc` y su exactitud de prueba en `full_tree_test_acc`.

In [ ]:
Xc_train, Xc_test, yc_train, yc_test = None, None, None, None   # TODO

full_tree = None              # TODO
full_tree_depth = None        # TODO: full_tree.get_depth()
full_tree_train_acc = None    # TODO
full_tree_test_acc = None     # TODO
print(f"profundidad {full_tree_depth} | entrenamiento {full_tree_train_acc} | prueba {full_tree_test_acc}")

In [ ]:
def check_3_2():
    if full_tree_test_acc is None:
        return _todo("3.2 todavía sin completar")
    a, b, c, d = train_test_split(X_churn, y_churn, test_size=0.25, stratify=y_churn, random_state=SEED)
    t = DecisionTreeClassifier(random_state=SEED).fit(a, c)
    if (full_tree_depth, round(full_tree_train_acc, 4), round(full_tree_test_acc, 4)) != \
       (t.get_depth(), round(t.score(a, c), 4), round(t.score(b, d), 4)):
        return _bad(f"se esperaba profundidad {t.get_depth()}, entrenamiento {t.score(a, c):.3f}, prueba {t.score(b, d):.3f}")
    _ok(f"3.2 correcto: exactitud de entrenamiento {full_tree_train_acc:.3f}, exactitud de prueba {full_tree_test_acc:.3f}")

check_3_2()

### 3.3 Elegir la profundidad (14 puntos)

1. Calcule una curva de validación con los datos de **entrenamiento** para `max_depth` de 1 a 20, con `cv=CV` y `scoring="accuracy"`. Guarde la exactitud **media** de entrenamiento y de validación a través de los pliegues en `tree_train_acc` y `tree_val_acc`.
2. Guarde en `churn_best_depth` la profundidad con la mayor exactitud de validación.
3. Ajuste el árbol con esa profundidad sobre todo el conjunto de entrenamiento y guarde su exactitud de prueba en `churn_final_test_acc`.

In [ ]:
depths = np.arange(1, 21)

tree_train_acc, tree_val_acc = None, None   # TODO
churn_best_depth = None                     # TODO
churn_final_test_acc = None                 # TODO
print("mejor profundidad:", churn_best_depth, "| exactitud de prueba:", churn_final_test_acc)

In [ ]:
if tree_val_acc is not None:
    fig, ax = plt.subplots()
    ax.plot(depths, tree_train_acc, "o-", color=BLUE, label="entrenamiento")
    ax.plot(depths, tree_val_acc, "s-", color=RED, label="validación")
    ax.axhline(churn_baseline, color="gray", ls=":", label="línea base (siempre «se queda»)")
    ax.axvline(churn_best_depth, color=GREEN, ls="--", label=f"mejor profundidad = {churn_best_depth}")
    ax.set(xlabel="max_depth", ylabel="exactitud", title="Caso 3: curva de validación del árbol de decisión")
    ax.legend(); plt.show()

    tuned = DecisionTreeClassifier(max_depth=churn_best_depth, random_state=SEED).fit(Xc_train, yc_train)
    summary = pd.DataFrame({
        "exactitud de entrenamiento": [np.nan, full_tree_train_acc, tuned.score(Xc_train, yc_train)],
        "exactitud de prueba": [churn_baseline, full_tree_test_acc, churn_final_test_acc]},
        index=["línea base (clase mayoritaria)", f"árbol sin restricciones (profundidad {full_tree_depth})",
               f"árbol ajustado (profundidad {churn_best_depth})"])
    display(summary.round(3))


def check_3_3():
    if churn_final_test_acc is None:
        return _todo("3.3 todavía sin completar")
    tr, va = validation_curve(DecisionTreeClassifier(random_state=SEED), Xc_train, yc_train,
                              param_name="max_depth", param_range=np.arange(1, 21), cv=CV, scoring="accuracy")
    if not np.allclose(tree_val_acc, va.mean(axis=1)):
        return _bad("tree_val_acc no coincide: ¿datos de entrenamiento, cv=CV y media sobre axis=1?")
    best = int(np.arange(1, 21)[np.argmax(va.mean(axis=1))])
    if churn_best_depth != best:
        return _bad(f"churn_best_depth debería ser {best}")
    exp = DecisionTreeClassifier(max_depth=best, random_state=SEED).fit(Xc_train, yc_train).score(Xc_test, yc_test)
    if not np.isclose(churn_final_test_acc, exp):
        return _bad(f"churn_final_test_acc debería ser {exp:.3f}")
    _ok(f"3.3 correcto: profundidad {best}, exactitud de prueba {exp:.3f}")

check_3_3()

### 3.4 Informe para la gerencia (8 puntos)

**Pregunta escrita.** Escriba un párrafo breve (de cinco a ocho oraciones) para una gerente sin formación técnica que:
- compare el árbol sin restricciones, el árbol ajustado y la línea base, usando los números de la tabla resumen;
- explique, sin tecnicismos, por qué el árbol que acierta el «100 %» con los clientes pasados no es el que se debe poner en producción;
- indique una limitación del análisis y un siguiente paso.

**Respuesta:** *(escriba aquí la respuesta)*

---
## Reflexión final (no calificada)

| Caso | Causa del sobreajuste | Remedio utilizado |
|---|---|---|
| 1. Helados | Grado del polinomio demasiado alto para 25 días de entrenamiento | Elegir el grado con una curva de validación |
| 2. Notas de examen | Demasiadas variables para 35 filas de entrenamiento | Conservar solo las variables con sentido; comparar con validación cruzada |
| 3. Abandono | Profundidad del árbol sin restricciones | Limitar `max_depth` con una curva de validación |

En los tres casos la **puntuación de entrenamiento** señalaba el modelo equivocado y la **puntuación de validación** señalaba el correcto. El conjunto de prueba se usó una sola vez, al final, para confirmar la elección.

### Lista de comprobación para la entrega

- [ ] Todas las celdas `check_...()` muestran ✅.
- [ ] Todas las respuestas escritas están completas.
- [ ] *Kernel → Restart & Run All* se ejecuta sin errores.